# Jev-Omni benchmark: re-analysis from the saved answers

Companion to [What Jev-Omni's decision head buys](https://condados.ai/blog/jev-omni-decision-head-benchmark).
This notebook needs no GPU. It reads the per-question answers in `output/results/` and recomputes
every accuracy, interval, calibration error and temperature the post quotes, then checks them
against `output/analysis.json`.

Arms: `jev` = Jev-Omni through its decision head (readout A), `jevdigits` = Jev-Omni's weights
through the LM-head digit readout (D), `base` = untouched Gemma 4 12B IT through the same readout (B).

In [ ]:
# Python 3.10+ standard library only; nothing to install.
import json, math, random, subprocess, sys
from pathlib import Path

import os
if not Path("output/results").exists() and Path("../output/results").exists():
    os.chdir("..")  # opened from notebooks/
if not Path("output/results").exists():  # on Colab: fetch the repository
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/CondadosAI/jev-omni-eval"], check=True)
    os.chdir("jev-omni-eval")

R = {}
for f in sorted(Path("output/results").glob("*__*.json")):
    bench, arm = f.stem.split("__")
    R[(bench, arm)] = json.loads(f.read_text())
print(len(R), "result files;", sum(len(v) for (b, a), v in R.items() if b in ("medium", "hard", "blink", "mmstar") and a == "base"), "questions per arm")

## Accuracy per arm (micro, per question)

In [ ]:
acc = lambda rows: 100 * sum(r["correct"] for r in rows) / len(rows)
for bench in ("medium", "hard", "blink", "mmstar"):
    print(f"{bench:7s}", "  ".join(f"{arm}={acc(R[(bench, arm)]):.2f}" for arm in ("jev", "jevdigits", "base")))

## McNemar's exact test on the disagreements

Only questions where the two readouts disagree carry information. Under equal skill each
disagreement is a fair coin.

In [ ]:
def mcnemar(a_rows, b_rows):
    A = {(r["id"], r.get("qkey")): r["correct"] for r in a_rows}
    B = {(r["id"], r.get("qkey")): r["correct"] for r in b_rows}
    a_only = sum(A[k] and not B[k] for k in A)
    b_only = sum(B[k] and not A[k] for k in A)
    n, k = a_only + b_only, min(a_only, b_only)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
    return a_only, b_only, p

for bench in ("medium", "hard", "blink", "mmstar"):
    a_only, b_only, p = mcnemar(R[(bench, "jev")], R[(bench, "base")])
    print(f"{bench:7s} head-only {a_only:3d}  base-only {b_only:3d}  p = {p:.5f}")

## 95% bootstrap interval of the paired accuracy difference (2,000 resamples, seed 0)

In [ ]:
def paired_ci(a_rows, b_rows, n_boot=2000, seed=0):
    B = {(r["id"], r.get("qkey")): r["correct"] for r in b_rows}
    keys = sorted(B)
    A = {(r["id"], r.get("qkey")): r["correct"] for r in a_rows}
    d = [int(A[k]) - int(B[k]) for k in keys]
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(d) for _ in d) / len(d) for _ in range(n_boot))
    return 100 * sum(d) / len(d), 100 * boots[int(0.025 * n_boot)], 100 * boots[int(0.975 * n_boot) - 1]

for arm in ("jev", "jevdigits"):
    for bench in ("medium", "hard", "blink", "mmstar"):
        diff, lo, hi = paired_ci(R[(bench, arm)], R[(bench, "base")])
        print(f"{arm:9s} - base  {bench:7s} {diff:+6.2f}  [{lo:+.2f}, {hi:+.2f}]")

## Expected calibration error and temperature scaling

ECE: ten equal-width bins on the top-option confidence. Temperature scaling divides every
log-probability by one constant T; the answer never changes, only the confidence. T is fitted by
negative log-likelihood on DecisionBench medium and applied unchanged everywhere else.

In [ ]:
def scaled(probs, T):
    lp = [math.log(max(p, 1e-300)) / T for p in probs]
    m = max(lp); e = [math.exp(x - m) for x in lp]; z = sum(e)
    return [x / z for x in e]

def ece_nll(rows, T=1.0):
    out = []
    for r in rows:
        p = scaled(r["probs"], T) if T != 1.0 else r["probs"]
        k = max(range(len(p)), key=p.__getitem__)
        out.append((p[k], k == r["gold"], max(p[r["gold"]], 1e-12)))
    bins = {}
    for c, ok, _ in out:
        bins.setdefault(min(int(c * 10), 9), []).append((c, ok))
    ece = sum(len(b) / len(out) * abs(sum(c for c, _ in b) / len(b) - sum(o for _, o in b) / len(b)) for b in bins.values())
    return ece, -sum(math.log(g) for *_, g in out) / len(out)

GRID = [round(0.25 + 0.05 * i, 2) for i in range(196)]
T = {arm: min(GRID, key=lambda t: ece_nll(R[("medium", arm)], t)[1]) for arm in ("jev", "jevdigits", "base")}
print("fitted T:", T)
for bench in ("medium", "hard", "blink", "mmstar"):
    print(f"{bench:7s}", "  ".join(f"{arm}: {ece_nll(R[(bench, arm)])[0]:.3f} -> {ece_nll(R[(bench, arm)], T[arm])[0]:.3f}" for arm in ("jev", "base")))

## The worked examples in the post

In [ ]:
# Top confidence bin of the untouched model on BLINK
rows = R[("blink", "base")]
top = [r for r in rows if r["confidence"] >= 0.9]
conf = sum(r["confidence"] for r in top) / len(top); hit = sum(r["correct"] for r in top) / len(top)
print(f"{len(top)} of {len(rows)} answers, mean confidence {conf:.3f}, accuracy {hit:.3f}, "
      f"ECE contribution {len(top) / len(rows) * abs(conf - hit):.3f}")

# BLINK Relative_Depth #47: every readout picks A, the answer is B
for arm in ("jev", "base"):
    r = next(r for r in R[("blink", arm)] if r["id"] == "val_Relative_Depth_47")
    gap = math.log(r["probs"][0] / r["probs"][1])
    print(f"{arm:4s} P(A) = {r['probs'][0]:.3f}  gap = {gap:.3f}  P_T(A) at T = {T[arm]} -> {1 / (1 + math.exp(-gap / T[arm])):.3f}")

## Check against `output/analysis.json`

In [ ]:
ref = json.loads(Path("output/analysis.json").read_text())
assert T == ref["T"], (T, ref["T"])
for bench in ("medium", "hard", "blink", "mmstar"):
    for arm in ("jev", "jevdigits", "base"):
        assert abs(round(ece_nll(R[(bench, arm)])[0], 4) - ref["benches"][bench][arm]["raw"]["ece"]) < 1e-4
        assert abs(round(ece_nll(R[(bench, arm)], T[arm])[0], 4) - ref["benches"][bench][arm]["scaled"]["ece"]) < 1e-4
    diff, lo, hi = paired_ci(R[(bench, "jev")], R[(bench, "base")])
    assert [round(lo, 2), round(hi, 2)] == ref["benches"][bench]["paired_jev_vs_base"]["micro_diff_ci95_pp"]
print("every checked number matches analysis.json")